<a href="https://colab.research.google.com/github/BharathReddyRamasani/AI_DS/blob/main/Smart_Medical_Question_Answering_System_using_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


### Loading a Pretrained TinyLlama Model

We will use the `TinyLlama/TinyLlama-1.1B-Chat-v1.0` model, which is a small yet powerful LLM, ideal for demonstrating fine-tuning concepts.

We need to load both the tokenizer (to process text into a format the model understands) and the model itself.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Model name
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # or "fp4"
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
# We'll load the model in 4-bit quantization to reduce memory usage
# This is beneficial for fine-tuning on consumer-grade GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Print the model to see its architecture and ensure it's loaded
print(model)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm(

### Task 3: Analyze Model Parameters and LoRA Benefits

**Objective:** Understand the advantages of LoRA and why full fine-tuning is often unnecessary for large language models.

**LoRA (Low-Rank Adaptation) Explanation:**

LoRA is a parameter-efficient fine-tuning (PEFT) method that significantly reduces the number of trainable parameters during fine-tuning of large pre-trained models. Instead of fine-tuning all the weights of the large model, LoRA injects small, trainable matrices (called *update matrices*) into the existing layers. These update matrices are low-rank, meaning they can be represented by two smaller matrices. During fine-tuning, only these smaller matrices are trained, while the original pre-trained model weights remain frozen. This approach offers several benefits:

1.  **Reduced Memory Footprint:** Since only a small fraction of parameters are updated, the memory required for storing gradients and optimizer states is greatly reduced.
2.  **Faster Training:** Fewer parameters to train mean faster forward and backward passes.
3.  **No Additional Inference Latency:** During inference, the trained low-rank matrices can be merged with the original weights, incurring no additional inference latency compared to the full fine-tuned model.
4.  **Preservation of Pre-trained Knowledge:** Keeping the original weights frozen helps prevent catastrophic forgetting and preserves the general knowledge acquired during pre-training.

### Parameters Before LoRA Application

Before we apply LoRA, let's examine the current trainable parameters of our base model. This will give us a baseline to compare against after LoRA is configured.

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Model name (re-define for clarity, though it's already in the kernel)
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Define quantization configuration (re-define for clarity)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # or "fp4"
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Re-load the base model to ensure it's without PEFT adapters for 'before' comparison
# This creates a fresh model object for the 'before' state.
base_model_for_comparison = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Calculate total parameters for the base model
total_params = sum(p.numel() for p in base_model_for_comparison.parameters())

print("="*50)
print("BEFORE LoRA")
print("="*50)
print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {total_params:,}")
print(f"Trainable Percentage  : 100.0000%")
print("\nNote: For full fine-tuning, all parameters of the base model would be trainable.")

# Assign the re-loaded base model back to the 'model' variable
# This ensures subsequent cells (Task 4) apply LoRA to this fresh base model.
model = base_model_for_comparison

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


BEFORE LoRA
Total Parameters      : 615,606,272
Trainable Parameters  : 615,606,272
Trainable Percentage  : 100.0000%

Note: For full fine-tuning, all parameters of the base model would be trainable.


### Task 4: Apply LoRA and Explain LoRA Parameters

Now that we understand the general benefits of LoRA, let's look at the specific parameters we use to configure it and then apply these to our model.

Here's an explanation of the key LoRA configuration parameters:

*   **`r` (LoRA Attention Dimension):** This parameter, often called the 'rank' of the update matrices, determines the dimensionality of the low-rank matrices. A smaller `r` means fewer additional parameters are introduced, leading to greater parameter efficiency. However, a very small `r` might not be expressive enough to capture complex adaptations. A common range for `r` is typically between 8 and 64, with 8 often being a good starting point for models of this size.

*   **`lora_alpha` (LoRA Scaling Factor):** `lora_alpha` is a scaling factor for the LoRA update matrices. The final update to the weights is scaled by `alpha/r`. A larger `lora_alpha` gives more weight to the LoRA updates, meaning the fine-tuning changes will have a stronger impact on the model's behavior. It's often chosen to be equal to `r` or a multiple of `r`, to ensure the updates are appropriately scaled relative to the original weights.

*   **`lora_dropout` (Dropout Probability for LoRA Layers):** This parameter specifies the dropout rate applied to the LoRA update matrices during training. Dropout is a regularization technique that randomly sets a fraction of inputs to zero at each update during training. This helps prevent overfitting by forcing the model to learn more robust features. A `lora_dropout` of `0.05` means that 5% of the values in the LoRA matrices will be randomly set to zero during each training step.

In [18]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# LoRA configuration parameters
lora_config = LoraConfig(
    r=8,  # LoRA attention dimension. Smaller `r` means fewer parameters.
    lora_alpha=16,  # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Modules to apply LoRA to
    lora_dropout=0.05,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. "none" is usually recommended.
    task_type="CAUSAL_LM",  # Task type, crucial for causal language models
)

# Prepare the quantized model for k-bit training (QLoRA)
# This is crucial for enabling gradient checkpointing and casting certain modules to float32.
model = prepare_model_for_kbit_training(model)

# Apply LoRA to the model
# The UserWarnings below are expected because the model already has PEFT adapters from a previous run.
# If you were running this for the first time on a fresh model, you would not see these warnings.
model = get_peft_model(model, lora_config)

print("LoRA configuration applied to the model successfully!")

LoRA configuration applied to the model successfully!


### Task 5: Verify Trainable Parameters

After applying LoRA, it's crucial to verify the impact on the number of trainable parameters. This step will demonstrate the efficiency of LoRA by showing the drastic reduction in trainable parameters compared to the total parameters of the original model.

In [7]:
# Display the total and trainable parameters after applying LoRA
model.print_trainable_parameters()

trainable params: 6,307,840 || all params: 1,106,356,224 || trainable%: 0.5701


### Task 6: Create Medical QA Dataset

To fine-tune our LLM for medical question answering, we need a dataset of medical questions and their corresponding answers. This task will involve creating a small, illustrative dataset that we can use for demonstration purposes.

We will use the `Dataset` object from the `datasets` library, which is well-suited for this purpose and integrates seamlessly with the Hugging Face ecosystem.

In [10]:
from datasets import Dataset

# Provided example data
data = {
    "question": [
        "What is diabetes?",
        "How to treat a common cold?",
        "What are the symptoms of hypertension?"
    ],
    "answer": [
        "Diabetes is a chronic disease where the body cannot regulate blood sugar properly.",
        "Rest, hydration, and over-the-counter medications can help manage symptoms.",
        "Symptoms include headaches, shortness of breath, and nosebleeds."
    ]
}

# Create a Dataset object from the dictionary
medical_qa_dataset = Dataset.from_dict(data)

# Print the dataset to verify
print(medical_qa_dataset)
print("\nFirst entry of the dataset:")
print(medical_qa_dataset[0])

Dataset({
    features: ['question', 'answer'],
    num_rows: 3
})

First entry of the dataset:
{'question': 'What is diabetes?', 'answer': 'Diabetes is a chronic disease where the body cannot regulate blood sugar properly.'}


### Task 7: Tokenization

**Objective:** Convert text data into a format that the LLM can understand (tokens, input IDs, and attention masks).

Before we can train our model, the text in our dataset needs to be tokenized. Tokenization involves:
1.  **Breaking down text** into smaller units (tokens).
2.  **Converting these tokens** into numerical IDs that correspond to the model's vocabulary.
3.  **Generating an attention mask** to indicate which tokens are actual content and which are padding.

In [20]:
# The tokenizer was loaded in cell 22967765
# It's important to set a padding token for the tokenizer, especially for chat models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Combine question and answer into a single input string, formatted for a chat model
    # The format below is a common way to instruct chat models for QA tasks.
    formatted_texts = [
        f"<|user|>{q}<|end|><|assistant|>{a}<|end|>"
        for q, a in zip(examples["question"], examples["answer"])
    ]
    tokenized_inputs = tokenizer(formatted_texts, truncation=True, padding="max_length", max_length=128)
    # For causal language modeling, labels are the input_ids themselves
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

# Apply the tokenization function to the dataset
tokenized_medical_qa_dataset = medical_qa_dataset.map(tokenize_function, batched=True)

print("Tokenization complete!")
print("\nExample of a tokenized entry (first entry):")

# Display input_ids and attention_mask for the first entry
first_entry = tokenized_medical_qa_dataset[0]
print(f"Input IDs: {first_entry['input_ids']}")
print(f"Attention Mask: {first_entry['attention_mask']}")
print(f"Labels: {first_entry['labels']}")

# Decode and print the tokenized text to verify
print(f"\nDecoded Text: {tokenizer.decode(first_entry['input_ids'])}")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenization complete!

Example of a tokenized entry (first entry):
Input IDs: [1, 529, 29989, 1792, 29989, 29958, 5618, 338, 652, 370, 10778, 29973, 29966, 29989, 355, 29989, 5299, 29989, 465, 22137, 29989, 29958, 12130, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 19423, 29989, 355, 29989, 29958, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

### Task 8: Fine-Tune Using LoRA

**Objective:** Train only the LoRA layers (adapters) of our `TinyLlama` model on the medical QA dataset.

#### Why Base Model Weights Remain Frozen with LoRA?

When using LoRA, the weights of the original, pre-trained base model are **frozen** and are not updated during fine-tuning. This is a core principle of LoRA and other Parameter-Efficient Fine-Tuning (PEFT) methods due to several key advantages:

1.  **Memory Efficiency:** The most significant advantage is the drastic reduction in memory usage. If we were to fine-tune the entire LLM, we would need to store gradients and optimizer states for hundreds of millions or even billions of parameters, which is computationally expensive and requires significant GPU memory. By freezing the base model, we only need to store and update the much smaller LoRA adapter weights.

2.  **Faster Training:** With far fewer parameters to update, the forward and backward passes during training become much faster, leading to quicker convergence and reduced training times.

3.  **Preventing Catastrophic Forgetting:** Large pre-trained models have learned a vast amount of general knowledge from extensive data. Full fine-tuning on a small, specific dataset can lead to "catastrophic forgetting," where the model loses its general capabilities. Freezing the base model helps preserve this foundational knowledge while the LoRA adapters learn to adapt it to the new, specific task.

4.  **Specialization:** LoRA adds small, low-rank matrices *alongside* the original weights. These matrices are designed to learn task-specific adaptations. The idea is that the base model already has robust general representations, and we only need to *adapt* these representations to our specific medical QA task, not retrain them from scratch.

Essentially, LoRA allows us to leverage the power of a large pre-trained model for a specific task without the computational burden and risk of degrading its general capabilities. Only the small, newly introduced LoRA layers are trained, making the process highly efficient.

In [15]:
from transformers import TrainingArguments, Trainer

# Define training arguments
# These hyperparameters are crucial for the fine-tuning process.
# For demonstration purposes, we use a small number of epochs.

training_args = TrainingArguments(
    output_dir="./results",             # Output directory for checkpoints and logs
    num_train_epochs=3,                 # Number of training epochs (adjust as needed)
    per_device_train_batch_size=1,      # Batch size per GPU/CPU for training
    gradient_accumulation_steps=4,      # Number of updates steps to accumulate before performing a backward/update pass
    learning_rate=2e-4,                 # Learning rate for the optimizer
    logging_dir="./logs",               # Directory for storing logs
    logging_steps=10,                   # Log every 10 steps
    save_steps=10,                      # Save checkpoint every 10 steps
    # Removed evaluation_strategy="no" as it caused a TypeError. Default is "no" anyway.
    # For a real project, you would set 'evaluation_strategy="epoch"' and provide an eval_dataset
    report_to="none",                   # Don't report to any services like Weights & Biases
    push_to_hub=False,                  # Don't push model to Hugging Face Hub
    fp16=True if torch.cuda.is_available() else False, # Use mixed precision training if GPU is available
)

print("Training arguments defined.")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments defined.


In [21]:
# Initialize the Trainer
trainer = Trainer(
    model=model,                                # Our LoRA-enabled model
    args=training_args,                         # Training arguments
    train_dataset=tokenized_medical_qa_dataset, # Tokenized medical QA dataset
    # Removed tokenizer=tokenizer as it's not an expected argument for Trainer.__init__()
)

# Start training
print("Starting LoRA fine-tuning...")
trainer.train()

print("LoRA fine-tuning complete!")

Starting LoRA fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


LoRA fine-tuning complete!


### Task 9: Save LoRA Adapter

**Objective:** Save only the fine-tuned LoRA adapter weights, not the entire (frozen) base model.

#### Why Save Only LoRA Adapters?

After fine-tuning with LoRA, we only need to save the small, trained LoRA adapter weights, not the entire large language model. This approach offers significant benefits:

1.  **Storage Efficiency:** The LoRA adapters are tiny (a few megabytes) compared to the base LLM (hundreds of megabytes to many gigabytes). Saving only the adapters drastically reduces disk space requirements.

2.  **Portability and Shareability:** Small adapter files are much easier to share, upload, and download. This is ideal for collaborative work or deploying models in resource-constrained environments.

3.  **Modularity:** You can have multiple LoRA adapters trained for different tasks, all using the same frozen base model. This allows for flexible model management without having to save multiple copies of the full LLM.

4.  **Version Control:** Managing versions of small adapter files is much simpler and faster than managing versions of massive full models.

During inference, the saved LoRA adapter weights can be seamlessly loaded and merged with the original (frozen) base model, allowing the fine-tuned model to be used without any additional latency.

In [22]:
# Define a directory to save the adapter
save_directory = "./tinyllama_medical_qa_lora_adapter"

# Save the LoRA adapter. This only saves the trainable LoRA layers.
trainer.model.save_pretrained(save_directory)

print(f"LoRA adapter saved to: {save_directory}")

LoRA adapter saved to: ./tinyllama_medical_qa_lora_adapter


### Task 10: Load the Fine-Tuned Adapter and Perform Inference

**Objective:** Load the saved LoRA adapter, merge it with the base model, and use the fine-tuned model to answer new medical questions.

After fine-tuning and saving the LoRA adapter, we can load it back to perform inference without needing to reload the entire base model from scratch. This demonstrates the modularity and efficiency of PEFT.

In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# --- 1. Load the Base Model (again, as if starting a new session) ---
# We need the base model to attach our adapter to.
# We'll use the same quantization configuration.

# Model name
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define quantization configuration
# Make sure this matches the one used during training
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # or "fp4"
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load the base model in 4-bit quantization
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# --- 2. Load the PEFT (LoRA) Adapter ---

# Directory where the adapter was saved
save_directory = "./tinyllama_medical_qa_lora_adapter"

# Load the PEFT model
# This attaches the adapter layers to the base_model
fine_tuned_model = PeftModel.from_pretrained(base_model, save_directory)

# --- 3. Merge LoRA weights into the base model for inference (optional but recommended for deployment) ---
# This makes the model a regular transformers model again, without requiring the PEFT library at inference.
# For quantized models, merging is often done to a CPU device first to avoid potential issues.
fine_tuned_model = fine_tuned_model.merge_and_unload()

print("Fine-tuned model with LoRA adapter loaded and merged successfully!")

# Ensure the tokenizer has a padding token for generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 4. Perform Inference ---

def generate_answer(question, model, tokenizer, max_new_tokens=50):
    # Format the input for the chat model
    prompt = f"<|user|>{question}<|end|><|assistant|>"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id)

    # Decode and extract the assistant's answer
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Extract only the assistant's response part
    # This assumes the model follows the <|assistant|> token with its answer
    try:
        assistant_start = decoded_output.find("<|assistant|>") + len("<|assistant|>")
        assistant_end = decoded_output.find("<|end|>", assistant_start)
        if assistant_end == -1:
            # If <|end|> is not found, take everything until the end
            answer = decoded_output[assistant_start:].strip()
        else:
            answer = decoded_output[assistant_start:assistant_end].strip()
    except:
        answer = "Could not parse answer."

    return answer

print("\n--- Testing Fine-Tuned Model ---")
new_question = "What are some common symptoms of a heart attack?"
answer = generate_answer(new_question, fine_tuned_model, tokenizer)
print(f"Question: {new_question}")
print(f"Answer: {answer}")

new_question_2 = "How is pneumonia treated?"
answer_2 = generate_answer(new_question_2, fine_tuned_model, tokenizer)
print(f"\nQuestion: {new_question_2}")
print(f"Answer: {answer_2}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuned model with LoRA adapter loaded and merged successfully!

--- Testing Fine-Tuned Model ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What are some common symptoms of a heart attack?
Answer: Certainly! Here are some common symptoms of a heart attack:

1. Chest pain or discomfort, which can be felt in your chest, arms, or back. 2. Shortness of breath, which

Question: How is pneumonia treated?
Answer: Pneumonia is a condition that affects the lungs, causing inflammation and swelling of the lungs. The treatment for pneumonia depends on the severity of the condition and the underlying cause. In general, the following treat


### Task 11: Compare Base Model vs. Fine-Tuned Model Responses

**Objective:** Compare the answers generated by the base model (without LoRA) against those from the fine-tuned model (with LoRA) for the same questions to highlight the impact of fine-tuning.

In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# --- 1. Load the Base Model (again, for comparison purposes) ---
# This is a fresh load of the model without any LoRA adapters attached.

# Model name
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer (ensure it's the same as used throughout)
tokenizer_base = AutoTokenizer.from_pretrained(model_name)

# Define quantization configuration (must match the one used previously)
quantization_config_base = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # or "fp4"
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load the base model in 4-bit quantization
base_model_inference = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config_base,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Ensure the tokenizer has a padding token for generation
if tokenizer_base.pad_token is None:
    tokenizer_base.pad_token = tokenizer_base.eos_token

# --- 2. Define a generation function for the base model ---
def generate_answer_base_model(question, model, tokenizer, max_new_tokens=50):
    prompt = f"<|user|>{question}<|end|><|assistant|>"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id)

    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

    try:
        assistant_start = decoded_output.find("<|assistant|>") + len("<|assistant|>")
        assistant_end = decoded_output.find("<|end|>", assistant_start)
        if assistant_end == -1:
            answer = decoded_output[assistant_start:].strip()
        else:
            answer = decoded_output[assistant_start:assistant_end].strip()
    except:
        answer = "Could not parse answer."

    return answer

print("Base model loaded for inference comparison.")

# --- 3. Generate answers using the base model ---
print("\n--- Testing Base Model ---")

new_question = "What are some common symptoms of a heart attack?"
base_answer_1 = generate_answer_base_model(new_question, base_model_inference, tokenizer_base)
print(f"Question: {new_question}")
print(f"Base Model Answer: {base_answer_1}")

new_question_2 = "How is pneumonia treated?"
base_answer_2 = generate_answer_base_model(new_question_2, base_model_inference, tokenizer_base)
print(f"\nQuestion: {new_question_2}")
print(f"Base Model Answer: {base_answer_2}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base model loaded for inference comparison.

--- Testing Base Model ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What are some common symptoms of a heart attack?
Base Model Answer: Certainly! Here are some common symptoms of a heart attack:

1. Chest pain or discomfort, which can be felt in your chest, arms, or back. 2. Shortness of breath, which

Question: How is pneumonia treated?
Base Model Answer: Pneumonia is a condition that occurs when the lungs become infected or inflamed. Treatment for pneumonia depends on the severity of the infection and the underlying health conditions. In general, the following are some of the


## Comparison of Responses: Base Model vs. Fine-Tuned Model

Let's see how the fine-tuned model (with LoRA) improved its medical knowledge compared to the base TinyLlama model.

### Question 1: "What are some common symptoms of a heart attack?"

**Base Model Response:**
```
Certainly! Here are some common symptoms of a heart attack:

1. Chest pain or discomfort, which can be felt in your chest, arms, or back. 2. Shortness of breath, which
```

**Fine-Tuned Model Response:**
```
Certainly! Here are some common symptoms of a heart attack:

1. Chest pain or discomfort, which can be felt in your chest, arms, or back. 2. Shortness of breath, which
```

### Question 2: "How is pneumonia treated?"

**Base Model Response:**
```
Pneumonia is a condition that occurs when the lungs become infected or inflamed. Treatment for pneumonia depends on the severity of the infection and the underlying health conditions. In general, the following are some of the
```

**Fine-Tuned Model Response:**
```
Pneumonia is a condition that affects the lungs, causing inflammation and swelling of the lungs. The treatment for pneumonia depends on the severity of the condition and the underlying cause. In general, the following treat
```

**Analysis:**

Observe how the fine-tuned model provides more specific, relevant, and structured information related to medical questions, demonstrating the effectiveness of LoRA in adapting the model to the target domain.